# 13 · Listing the legal moves in the prompt (`sft_legal_moves`)

**The question.** The largest single failure of our SFT model is legality: 28% of `sft_v1`'s answers on the test puzzles are illegal moves. The model has to work out from the FEN text alone which moves are possible, and often names a move that is not. This experiment asks: **if the prompt lists every legal move, so the model only has to choose one, does it play better?**

**What changes, and what stays the same.** Only the prompt changes. One line is added before the question:

```
Legal moves: a2a3 a2a4 b1a3 b1c3 ...
```

Everything else is identical to `sft_v1`: the same 20,000 training positions in the same order, the same answers (the move the player made), and the same settings. Training and grading both use the new prompt, as every model is always asked the same way it was trained.

**Two models are graded:**
- `base_legal_moves`: the base model, untrained, with the new prompt. Compared with `base`, this shows what the list alone does.
- `sft_legal_moves`: SFT on the same 20,000 positions with the new prompt. Compared with `sft_v1`, this is the effect of the list on a trained model.

**What to keep in mind.** The list gives the model information the other models must work out themselves, so this is a different, easier version of the task, not a fix for the original one. It is still useful: it separates two reasons a model can fail ("names an impossible move" and "chooses a poor move among the possible ones"). If legality rises but puzzle accuracy does not, the models' problem is choosing, not knowing the rules.

**Running it on Databricks**
1. Open this notebook in the shared Git folder and pick **Serverless GPU → Small (1×A10)**. Plain *Serverless* has no GPU. Medium and Large cost much more and are not needed here.
2. Set `QUICK_CHECK = True` in the Settings cell and click **Run all**. This takes a few minutes, uses tiny sizes, and saves nothing to `results/`. It catches setup problems before the long run. (The first cell restarts Python once; if Run all stops there, click Run all again.)
3. Set `QUICK_CHECK = False` and run the real thing as a **Job**, so the browser can close: New → Job → type *Notebook* → this notebook → compute *Serverless GPU, 1×A10*; add an email notification. Separate Jobs can run at the same time.
4. Afterwards, commit and push **only `results/`** from the Git folder. Leave `notebooks/` unticked: Databricks cannot save notebook outputs to git, so committing a notebook from there would erase its outputs on GitHub. The outputs of a Job run are on the Job's run page; every figure and table is also saved in `results/figures/`.

**Time:** about 1.5–2 hours on an A10 (the prompts are about twice as long as before, so training is slower than `sft_v1`'s 20,000 examples), roughly $8–10 of credits.

**Outputs**
- the adapter in `checkpoints/sft_legal_moves/` (not on GitHub)
- results: `results/sft_legal_moves_training.json`, `results/sft_legal_moves.json`, `results/base_legal_moves.json` (and their `_dev` files), and rows in `results/summary.csv`
- every figure and table of this experiment in `results/figures/sft_legal_moves/`

In [ ]:
# On Databricks: install our packages and Stockfish for this session. Does nothing on other machines.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # so Databricks finds our code in src/
from chesslm.databricks import setup_databricks

setup_databricks()

In [ ]:
# Settings
RUN_NAME = "sft_legal_moves"
BASE_NAME = "base_legal_moves"
QUICK_CHECK = False   # True = a few-minute run with tiny sizes, to check that everything works. Saves nothing to results/.

import gc
import math

import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.width", 200)  # print wide tables on one line

from chesslm import experiments, plots
from chesslm.config import REPO_ROOT, load_config
from chesslm.evaluate import load_eval_model
from chesslm.experiments import compare_with, figures_dir, grade_puzzles, puzzle_table, save_table, save_training
from chesslm.model import load_tokenizer
from chesslm.prompts import build_messages
from chesslm.train_sft import sample_examples, to_dataset, train

sft_cfg = load_config("sft")
prompt_cfg, eval_cfg = load_config("prompt"), load_config("eval")
old_prompt_cfg = dict(prompt_cfg)
prompt_cfg["show_legal_moves"] = True   # THE change in this experiment; everything else is as in sft_v1
configs = {"model": sft_cfg["model"], "prompt": prompt_cfg, "eval": eval_cfg}
SEED = sft_cfg["training"]["seed"]

if QUICK_CHECK:
    experiments.use_quick_check_folder()
    RUN_NAME += "_check"
    BASE_NAME += "_check"
    sft_cfg["data"].update(trial_examples=16, train_examples=64, val_examples=16)
    sft_cfg["training"].update(logging_steps=1, eval_steps=1)
PUZZLE_LIMIT = 5 if QUICK_CHECK else None
ADAPTER_DIR = REPO_ROOT / "checkpoints" / RUN_NAME
FIGURES = figures_dir(RUN_NAME)
print(f"Run: {RUN_NAME} | legal moves in the prompt: {prompt_cfg['show_legal_moves']} | figures and tables -> {FIGURES}")

## 1. The new prompt

The same 20,000 training positions as `sft_v1`. Below, one position with the old and the new prompt.

The list makes prompts longer. Training cuts every example to a maximum length, and an example cut short would lose its answer, so the cell measures the longest example and sets the maximum length above it.

In [ ]:
n = sft_cfg["data"]
train_rows = sample_examples(pd.read_parquet(REPO_ROOT / "data" / "processed" / "train.parquet"), n["train_examples"], SEED)
val_rows = sample_examples(pd.read_parquet(REPO_ROOT / "data" / "processed" / "val.parquet"), n["val_examples"], SEED)
row = train_rows.iloc[0].to_dict()
print("Old prompt:\n" + build_messages(row, old_prompt_cfg)[0]["content"] + "\n")
print("New prompt:\n" + build_messages(row, prompt_cfg)[0]["content"])

tokenizer = load_tokenizer(sft_cfg["model"])
def length(row, cfg):
    messages = build_messages(row, cfg) + [{"role": "assistant", "content": row[f"move_{cfg['notation']}"]}]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return len(tokenizer(text, add_special_tokens=False).input_ids)
rows = pd.concat([train_rows, val_rows]).to_dict("records")
old_lengths = pd.Series([length(r, old_prompt_cfg) for r in rows[:2000]])
new_lengths = pd.Series([length(r, prompt_cfg) for r in rows])
sft_cfg["training"]["max_seq_length"] = max(256, math.ceil(new_lengths.max() / 64) * 64)
lengths = pd.DataFrame({"Average length (tokens)": [round(old_lengths.mean()), round(new_lengths.mean())],
                        "Longest (tokens)": [old_lengths.max(), new_lengths.max()]},
                       index=pd.Index(["old prompt (sft_v1)", "with legal moves"], name="Prompt"))
save_table(lengths, FIGURES, "prompt_length",
           "Length of a training example (prompt and answer) in tokens. The old prompt is measured on 2,000 examples.")
display(lengths)
print(f"Maximum training length set to {sft_cfg['training']['max_seq_length']} tokens, so no example is cut short.")
del tokenizer

## 2. The base model with the new prompt (no training)

Before any training: does the list alone change what the base model answers? Graded like every other model: 200 dev puzzles, then the 1,000 test puzzles.

In [ ]:
tokenizer, model = load_eval_model(sft_cfg["model"])
grade_puzzles(model, tokenizer, "dev", f"{BASE_NAME}_dev", configs, None, PUZZLE_LIMIT)
grade_puzzles(model, tokenizer, "test", BASE_NAME, configs, None, PUZZLE_LIMIT)
del model
gc.collect()
torch.cuda.empty_cache()

## 3. Trial run

A short run on the first few hundred training examples, to check that the loss goes down on this machine before the long run.

In [ ]:
trial_ds = to_dataset(train_rows.head(n["trial_examples"]), prompt_cfg)
val_ds = to_dataset(val_rows, prompt_cfg)
trial_log, trial_compute = train(trial_ds, val_ds.select(range(min(100, len(val_ds)))), sft_cfg,
                                 REPO_ROOT / "checkpoints" / f"{RUN_NAME}_trial")
losses = trial_log.loss.dropna()
print(f"Training loss: {losses.iloc[0]:.3f} at the start -> {losses.iloc[-1]:.3f} at the end")
gc.collect()
torch.cuda.empty_cache()

## 4. Full training run

The same settings as `sft_v1`, with the new prompt. Both losses should fall and level off.

In [ ]:
train_ds = to_dataset(train_rows, prompt_cfg)
print(f"train: {len(train_ds):,} examples  validation: {len(val_ds):,} examples")
log, compute = train(train_ds, val_ds, sft_cfg, ADAPTER_DIR)
save_training(RUN_NAME, {"sft": sft_cfg, "prompt": prompt_cfg}, compute, log)
plots.loss_curve(log, FIGURES / "training_loss.png", title=f"{RUN_NAME}: loss during training");

## 5. Grade the model on the puzzles

The same grader, puzzles, and greedy decoding as every other model, with the new prompt.

In [ ]:
gc.collect()
torch.cuda.empty_cache()
tokenizer, model = load_eval_model(sft_cfg["model"], adapter_path=ADAPTER_DIR)
grade_puzzles(model, tokenizer, "dev", f"{RUN_NAME}_dev", configs, str(ADAPTER_DIR), PUZZLE_LIMIT)
grade_puzzles(model, tokenizer, "test", RUN_NAME, configs, str(ADAPTER_DIR), PUZZLE_LIMIT)
del model

## 6. Results: does the list help?

**How to read this section**
- **Legal (%)** should rise with the list. The question is whether **puzzle accuracy** rises with it.
- **Accuracy when legal** is puzzle accuracy among the legal answers only. If it stays the same while legality rises, the list removed impossible moves without improving the choice among possible ones.
- The **paired comparisons** count puzzles solved by only one of two models (p below 0.05 = unlikely to be chance): `base_legal_moves` vs. `base` is the effect of the list without training, `sft_legal_moves` vs. `sft_v1` the effect with training.

All numbers are on the 1,000 test puzzles; a random legal move solves 5.6%.

In [ ]:
runs = ["base", BASE_NAME, "sft_v1", RUN_NAME]
table = puzzle_table(runs, FIGURES)
display(table)

choice = table[["Puzzle accuracy (%)", "Legal (%)"]].copy()
choice["Accuracy when legal (%)"] = (choice["Puzzle accuracy (%)"] / choice["Legal (%)"] * 100).round(1)
save_table(choice, FIGURES, "accuracy_when_legal",
           "Accuracy when legal = puzzle accuracy among the legal answers only (accuracy / legal rate).")
display(choice)

comparison = pd.DataFrame([compare_with(BASE_NAME, "base"), compare_with(RUN_NAME, "sft_v1"),
                           compare_with(RUN_NAME, BASE_NAME)]).set_index("model")
save_table(comparison, FIGURES, "paired_comparison",
           "Puzzles solved by only one of the two models, and the sign-test p-value (below 0.05 = unlikely to be chance).")
display(comparison)

In [ ]:
summary = pd.read_csv(experiments.RESULTS / "summary.csv")
plots.metric_panels(choice, [(c, c, "{:.1f}") for c in choice.columns], FIGURES / "accuracy_when_legal.png",
                    title="Does listing the legal moves help? (test puzzles)");
plots.outcome_breakdown(summary, runs, FIGURES / "answer_outcomes.png", title="Are the answers legal moves?");
plots.accuracy_by_band(summary, runs, FIGURES / "accuracy_by_band.png",
                       title="Puzzle accuracy by puzzle difficulty (rating band)");

## Summary of outputs

- `results/figures/sft_legal_moves/`: `prompt_length` (table), `training_loss.png`, `puzzle_results` (table), `accuracy_when_legal` (table and figure), `paired_comparison` (table), `answer_outcomes.png`, `accuracy_by_band.png`
- `results/sft_legal_moves.json`, `results/base_legal_moves.json`: every test answer and its grade; `results/sft_legal_moves_training.json`: settings, compute, training log
- Both models are also graded on held-out game positions and in full games in notebook 11, with the new prompt.
- If listing the moves helps, notebook 15 can combine it with more data (notebook 12).